<div style="background:linear-gradient(135deg, rgba(249,112,102,0.18), rgba(249,112,102,0.02)); border-left:6px solid #F97066; border-radius:10px; padding:20px 24px; margin-bottom:20px;">
<h1 style="margin:0; color:#F97066; font-size:1.8em;">⏳ Python Asíncrono</h1>
<p style="margin:6px 0 0; opacity:0.8;">Unidad 1 — Programación asíncrona con <code>asyncio</code>, <code>async</code> y <code>await</code></p>
</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:0 0 24px; background:rgba(14,165,233,0.04);">
<strong>📑 Contenido de esta guía</strong>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><a href="#corrutinas">Corrutinas y <code>await</code></a></li>
<li><a href="#secuencial-vs-paralelo">Ejecución secuencial vs. paralela</a></li>
<li><a href="#generadores-asincronos">Generadores asíncronos (streaming)</a></li>
<li><a href="#fuera-de-jupyter">Uso fuera de Jupyter</a></li>
<li><a href="#cierre">Cierre y próximos pasos</a></li>
</ol>
</div>

La programación **asíncrona** permite que un programa siga trabajando mientras espera algo "lento" (una respuesta de red, leer un archivo, esperar a una API de un LLM), en lugar de quedarse bloqueado sin hacer nada. En Python esto se logra con el módulo `asyncio`, las funciones `async def` y la palabra clave `await`.

A diferencia del *multithreading*, `asyncio` usa un único hilo con un **bucle de eventos** (*event loop*) que va alternando entre tareas cada vez que una de ellas queda en espera. Es ideal para tareas de I/O (red, disco, APIs), no para cálculos pesados de CPU.

<div style="border-left:4px solid #F97066; background:rgba(249,112,102,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>⚠️ Nota importante</strong><br>
En varias celdas de este notebook verá <code>await</code> escrito directamente en una celda, fuera de cualquier función. Esto <strong>solo funciona en Jupyter/IPython</strong>, porque estos entornos ya mantienen un bucle de eventos corriendo por detrás. En un script <code>.py</code> normal, escribir <code>await</code> fuera de una función <code>async def</code> produce un <code>SyntaxError</code>; ahí se debe envolver el código en una función <code>async def main(): ...</code> y ejecutarla con <code>asyncio.run(main())</code>.
</div>

---

<a id="corrutinas"></a>

## <span style="color:#F97066;">Corrutinas y `await`</span>

A continuación, un primer ejemplo de función asíncrona.

In [1]:
# Definamos una función asíncrona

import asyncio 

async def do_some_work():
    print("Iniciando trabajo")
    await asyncio.sleep(1)
    print("Trabajo completado")

Con `async def` se define una **corrutina**: una función especial que puede "pausarse" en cada `await` sin bloquear el resto del programa. `asyncio.sleep(1)` simula una espera de 1 segundo (como si fuera una llamada de red), cediendo el control al bucle de eventos mientras espera.

¿Qué ocurre si se llama a `do_some_work()` como si fuera una función normal, sin `await`?

In [2]:
# ¿Qué hará esto?

do_some_work()

<coroutine object do_some_work at 0x10d0913c0>

No imprime "Iniciando trabajo" ni "Trabajo completado". Llamar a una función `async def` sin `await` **no la ejecuta**: solo crea un objeto `coroutine` (por eso el resultado es algo como `<coroutine object do_some_work at 0x...>`). Para que el código dentro realmente se ejecute, hay que usar `await`.

In [3]:
# ¡OK, intentémoslo de nuevo!

await do_some_work()

Iniciando trabajo
Trabajo completado


Ahora se ejecuta correctamente. A continuación, un error común: llamar a varias corrutinas dentro de otra función `async` **sin `await`** en cada una.

<a id="secuencial-vs-paralelo"></a>

## <span style="color:#F97066;">Ejecución secuencial vs. paralela</span>

In [4]:
# ¿Qué está mal con esto?

async def do_a_lot_of_work():
    do_some_work()
    do_some_work()
    do_some_work()

await do_a_lot_of_work()

/var/folders/4s/h_w2hwwd7pg468h5wgcvnlfr0000gn/T/ipykernel_48163/2691356138.py:4: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()
/var/folders/4s/h_w2hwwd7pg468h5wgcvnlfr0000gn/T/ipykernel_48163/2691356138.py:5: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()
/var/folders/4s/h_w2hwwd7pg468h5wgcvnlfr0000gn/T/ipykernel_48163/2691356138.py:6: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()


Este código se ejecuta casi instantáneamente y no imprime nada de `do_some_work()`, además de mostrar advertencias `RuntimeWarning: coroutine 'do_some_work' was never awaited`. Es el mismo error que antes, repetido tres veces: cada llamada crea una corrutina que nunca se ejecuta porque le falta `await`. Corrijámoslo.

In [5]:
# ¡Advertencia interesante! Corrijámoslo

async def do_a_lot_of_work():
    await do_some_work()
    await do_some_work()
    await do_some_work()

await do_a_lot_of_work()

Iniciando trabajo
Trabajo completado
Iniciando trabajo
Trabajo completado
Iniciando trabajo
Trabajo completado


Ahora sí se ejecutan las tres llamadas, pero **una tras otra**: como cada `await` espera a que termine la anterior antes de seguir, el total tarda ~3 segundos (3 × 1 segundo). Esto sigue siendo secuencial, no concurrente.

### Ejecución en paralelo con `asyncio.gather()`

Para que las tres corran *al mismo tiempo*, se utiliza `asyncio.gather()`.

In [6]:
# Y ahora hagámoslo en paralelo
# Es importante reconocer que esto no es "multithreading" de la manera a la que podría estar acostumbrado.
# La librería asyncio se ejecuta en un solo hilo, pero utiliza un bucle de eventos para cambiar entre tareas mientras una está en espera.

async def do_a_lot_of_work_in_parallel():
    await asyncio.gather(do_some_work(), do_some_work(), do_some_work())

await do_a_lot_of_work_in_parallel()

Iniciando trabajo
Iniciando trabajo
Iniciando trabajo
Trabajo completado
Trabajo completado
Trabajo completado


Ahora las tres tareas inician casi al mismo tiempo ("Iniciando trabajo" se imprime 3 veces seguidas) y el total tarda ~1 segundo en vez de 3: mientras una está en `await asyncio.sleep(1)`, el bucle de eventos aprovecha para avanzar las otras dos. Esta es la base de por qué `asyncio` es útil para hacer varias llamadas de red o a APIs (por ejemplo, a un LLM) en paralelo sin usar hilos.

<a id="fuera-de-jupyter"></a>

## <span style="color:#F97066;">Uso fuera de Jupyter</span>

Fuera de Jupyter, este mismo ejemplo se estructura como:

```python
import asyncio

async def main():
    await asyncio.gather(do_some_work(), do_some_work(), do_some_work())

if __name__ == "__main__":
    asyncio.run(main())
```

---

<a id="generadores-asincronos"></a>

# <span style="color:#F97066;">Generadores asíncronos: simulando streaming</span>

En el notebook <code>3_python_intermedio.ipynb</code> se presentaron los generadores (funciones con `yield`). Es posible combinar esa idea con `async`/`await` para crear un **generador asíncrono**: una función que produce valores de uno en uno, cediendo el control al bucle de eventos entre cada uno. Esto es exactamente lo que ocurre cuando un modelo de lenguaje devuelve su respuesta en *streaming*, token a token.

Un generador asíncrono se define con `async def` y `yield`, y se recorre con `async for` en lugar de `for`.

<span style="background:#0EA5E9;color:#ffffff;padding:4px 14px;border-radius:14px;font-size:0.75em;">🧪 EJERCICIO</span>

Ejecute la siguiente celda y observe cómo cada palabra aparece de forma progresiva, simulando la salida de un LLM en tiempo real.

In [7]:
# Un generador asíncrono: usa `yield` dentro de una función `async def`

async def stream_tokens(texto):
    for palabra in texto.split():
        await asyncio.sleep(0.3)  # simula la latencia de red de un LLM real
        yield palabra

# Se recorre con `async for`, no con `for`

async def mostrar_stream():
    async for token in stream_tokens("Los modelos de lenguaje generan texto token a token"):
        print(token, end=" ", flush=True)

await mostrar_stream()

Los modelos de lenguaje generan texto token a token 

---

<a id="cierre"></a>

# <span style="color:#F97066;">🎯 Cierre y próximos pasos</span>

<div style="border-left:4px solid #14B8A6; background:rgba(20,184,166,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>✅ Resumen</strong><br>
En este notebook se presentaron los siguientes conceptos de programación asíncrona:

- Corrutinas definidas con <code>async def</code> y ejecutadas con <code>await</code>.
- La diferencia entre ejecución secuencial (<code>await</code> uno tras otro) y paralela (<code>asyncio.gather()</code>).
- Generadores asíncronos (<code>async def</code> + <code>yield</code>, recorridos con <code>async for</code>) para simular streaming.
- Cómo estructurar este mismo código en un script <code>.py</code> fuera de Jupyter, con <code>asyncio.run(main())</code>.

</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>➡️ Continúe con</strong>
<ul style="margin:8px 0 0; padding-left:20px;">
<li><code>5_pydantic.ipynb</code> — validación de datos con Pydantic.</li>
</ul>
</div>